# Transformers and LLM — Homework

Before running the notebook (once, in a terminal):

```bash
ollama serve # must run in the background; on macOS, opening Ollama.app is enough
ollama pull qwen2.5:3b # compact tool-capable model (~1.9 GB) for a Mac with 8 GB RAM
ollama run qwen2.5:3b # optional quick check; press Ctrl+D to exit
```

## Setup - local client and agent loop

In [ ]:
import json
from openai import OpenAI

OLLAMA_MODEL = "qwen2.5:3b" # compact model with tool support for 8 GB RAM
ollama_client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

# Smoke test
r = ollama_client.chat.completions.create(
    model=OLLAMA_MODEL,
    messages=[{"role": "user", "content": "Reply with one word: ready"}])
print("Model says:", r.choices[0].message.content)

def run_agent_local(user_prompt, tools, dispatch,
                    client=ollama_client, model=OLLAMA_MODEL, max_steps=5):
    #messages = [{"role": "user", "content": user_prompt}] change this prompt to more advanced one:
    messages = [
    {
        "role": "system",
        "content": (
            "Call exactly one tool per turn. "
            "Wait for the tool result before calling the next tool. "
            "Use calculator for all arithmetic. "
            "Never invent values."
        ),
    },
    {
        "role": "user",
        "content": user_prompt,
    },
]

    for step in range(max_steps):
        response = client.chat.completions.create(
            model=model, messages=messages, tools=tools, tool_choice="auto")
        message = response.choices[0].message
        messages.append(message.model_dump(exclude_none=True))

        if not message.tool_calls:
            return message.content

        for tool_call in message.tool_calls:
            args = json.loads(tool_call.function.arguments)
            print(f"[step {step}] {tool_call.function.name}({args})")
            messages.append({"role": "tool",
                             "tool_call_id": tool_call.id,
                             "content": json.dumps(dispatch[tool_call.function.name](**args))})

    return "Stopped: max_steps reached."

Model says: Ready.


## Task 1 — a third tool and a chain of calls

In [ ]:
def get_stock_price(ticker: str):
    prices = {"NVDA": 125.50, "GOOG": 178.20, "AAPL": 229.00}
    return {"ticker": ticker.upper(), "price_usd": prices.get(ticker.upper(), 0.0)}

def calculator(expression: str):
    return {"expression": expression, "result": eval(expression, {"__builtins__": {}}, {})}

# TODO 1: finish the third tool — convert a USD amount to another currency.
def convert_currency(amount_usd: float, to: str):
    rates = {"EUR": 0.92, "GBP": 0.79, "UAH": 41.0}
    code = to.upper()
    if code not in rates:
        return {"error": f"unsupported currency: {to}", "supported": list(rates)}
    return {"amount": round(amount_usd * rates[code], 2), "currency": code}

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_stock_price",
            "description": (
                "Get the USD price of ONE share for a ticker symbol. "
                "If the user requests multiple shares, pass this price to "
                "the calculator tool. Do not calculate the total yourself."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "ticker": {
                        "type": "string",
                        "description": "Stock ticker symbol, e.g. NVDA",
                    }
                },
                "required": ["ticker"],
            },
        },
    },

    {
        "type": "function",
        "function": {
            "name": "calculator",
            "description": (
                "Perform all arithmetic operations required by the user. "
                "This tool must be used to multiply a stock price by the "
                "number of shares. Use literal numbers, for example "
                "'125.5 * 10'. Do not calculate arithmetic mentally."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": (
                            "Arithmetic expression containing literal numbers, "
                            "for example '125.5 * 10'."
                        ),
                    }
                },
                "required": ["expression"],
            },
        },
    },
# TODO 2: add the function spec for convert_currency here
    {
        "type": "function",
        "function": {
            "name": "convert_currency",
            "description": (
                "Convert a USD total into EUR, GBP or UAH. "
                "For calculations involving multiple shares, amount_usd must "
                "be the result returned by the calculator tool. "
                "Do not calculate the USD total inside this tool call."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "amount_usd": {
                        "type": "number",
                        "description": (
                            "USD total returned by the calculator tool, "
                            "for example 1255.0."
                        ),
                    },
                    "to": {
                        "type": "string",
                        "enum": ["EUR", "GBP", "UAH"],
                        "description": "Target currency code.",
                    },
                },
                "required": ["amount_usd", "to"],
            },
        },
    },
]

# TODO 3: register the new tool so the loop can dispatch to it
DISPATCH = {
    "get_stock_price": get_stock_price,
    "calculator": calculator,
    "convert_currency": convert_currency,
}
# TODO 4: ask something that needs all three tools, then run it:

prompt = ("How much are 10 shares of NVDA worth in euros? ")

print(run_agent_local(prompt, TOOLS, DISPATCH))

[step 0] get_stock_price({'ticker': 'NVDA'})
[step 1] calculator({'expression': '125.5 * 10'})
[step 2] convert_currency({'amount_usd': 1255, 'to': 'EUR'})
The 10 shares of NVDA are worth approximately 1154.60 Euros.


**▶️ Expected output** (after completing TODO 1–4 and uncommenting the last line)

```
  [step 0] -> get_stock_price({'ticker': 'NVDA'})
  [step 1] -> calculator({'expression': '125.5 * 10'})
  [step 2] -> convert_currency({'amount_usd': 1255.0, 'to': 'EUR'})
10 shares of NVDA are worth about €1,154.60 (1255 USD × 0.92).
```

The model chained all three tools on its own. Exact wording, the order of the middle steps, and rounding will vary between runs and models — what matters is that **all three tools get called** and the final number is right.

## Task 2 — structured output locally

JSON mode guarantees *valid* JSON but does not guarantee *which fields* it contains. Here a 
JSON Schema generated from a Pydantic class through `format=`, and Ollama constrains generation
to that schema. `model_validate_json` is the final safeguard: if the response does not match the
schema, it raises an exception instead of silently accepting corrupted data.

In [4]:
import ollama
from pydantic import BaseModel, Field
from typing import Literal

# TODO 1: define the schema
class Review(BaseModel):
    sentiment: Literal["positive", "negative", "mixed"]
    score: Literal[1, 2, 3, 4, 5]
    pros: list[str]
    cons: list[str]

text = ("Battery life is fantastic and it is super light, but the camera is "
        "mediocre in low light and the price is a bit high.")

prompt = ("Extract this product review as JSON. "
          "sentiment must be one of: positive, negative, mixed. "
          "score is an integer from 1 to 5.\n"
          f"Review: {text}")

# TODO 2: call ollama with format=<your schema>, then validate the JSON reply.
response = ollama.chat(
    model=OLLAMA_MODEL,
    messages=[{"role": "user", "content": prompt}],
    format=Review.model_json_schema(), # use a schema instead of unconstrained text
    options={"temperature": 0},
)

raw = response.message.content
review = Review.model_validate_json(raw) # raises if the response does not match the schema

print(review)
print()
print("sentiment:", review.sentiment)
print("score:", review.score)
print("pros:", review.pros)
print("cons:", review.cons)

sentiment='positive' score=4 pros=['Battery life is fantastic', 'It is super light'] cons=['Camera is mediocre in low light', 'Price is a bit high']

sentiment: positive
score: 4
pros: ['Battery life is fantastic', 'It is super light']
cons: ['Camera is mediocre in low light', 'Price is a bit high']


**▶️ Expected output** (after completing TODO 1–2)

```
sentiment='mixed' score=3 pros=['fantastic battery life', 'lightweight'] cons=['mediocre low-light camera', 'price a bit high']
```

`review` is a real, validated `Review` object — the JSON came back matching your schema exactly. The phrasing of the pros/cons will differ per run; the structure won't.

## Task 3 — extending the MCP server

Add a second tool, `get_air_quality`, to the server. **No LLM is involved here**; test the
protocol itself. The client starts the server as a subprocess, performs the `initialize` handshake,
calls `list_tools()`, and then uses `call_tool()`. Never import `get_air_quality` directly;
*discover* it through the protocol.

In [5]:
%%writefile weather_server_hw.py
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("weather-hw")

@mcp.tool()
def get_forecast(city: str) -> str:
    """Return a short weather forecast for a city."""
    data = {"Kyiv": "18C, partly cloudy",
            "London": "12C, rain",
            "Tokyo": "24C, clear"}
    return data.get(city, f"No forecast available for {city}")

# TODO 1: add a get_air_quality(city) tool that returns an AQI string per city.
@mcp.tool()
def get_air_quality(city: str) -> str:
    """Return the air-quality index (AQI) for a city."""
    data = {"Kyiv": "AQI 42 (good)",
            "London": "AQI 58 (moderate)",
            "Tokyo": "AQI 35 (good)"}
    return data.get(city, f"AQI unknown for {city}")

if __name__ == "__main__":
    mcp.run(transport="stdio")

Overwriting weather_server_hw.py


**▶️ Expected output**

```
Writing weather_server_hw.py
```

(`Overwriting weather_server_hw.py` if you run it again.) The cell just saves the file — it does **not** start the server here.

In [6]:
import sys
from pathlib import Path
from tempfile import TemporaryFile

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

# sys.executable is the same Python interpreter that runs this notebook.
# On macOS, the command "python" may be missing or may point to a Python without mcp.
server_file = Path("weather_server_hw.py").resolve()
server_hw = StdioServerParameters(command=sys.executable, args=[str(server_file)])

async def check_weather_server():
    # Redirect server stderr so it does not clutter the notebook output.
    with TemporaryFile("w+", encoding="utf-8") as errlog:
        async with stdio_client(server_hw, errlog=errlog) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()

                tools = await session.list_tools()
                print("Discovered tools:", [t.name for t in tools.tools])
                # TODO 2: call get_air_quality for a city and print result.content[0].text

                forecast = await session.call_tool("get_forecast", {"city": "Kyiv"})
                air = await session.call_tool("get_air_quality", {"city": "Kyiv"})

                print("Kyiv forecast:", forecast.content[0].text)
                print("Kyiv air quality:", air.content[0].text)

await check_weather_server() # in a regular .py file: asyncio.run(check_weather_server())

Discovered tools: ['get_forecast', 'get_air_quality']
Kyiv forecast: 18C, partly cloudy
Kyiv air quality: AQI 42 (good)


**▶️ Expected output** (after completing TODO 2)

```
Discovered tools: ['get_forecast', 'get_air_quality']
Kyiv air quality: AQI 42 (good)
```

Both tools are now visible over the protocol, and the new one returns your data — all with no LLM involved.

## Task 4 — reading

- MCP intro: https://modelcontextprotocol.io
- Hugging Face, fine-tuning for text classification: https://huggingface.co/docs/transformers/tasks/sequence_classification

### Stretch — local model with MCP tools

Сombine Task 1 and Task 3. The agent loop stays the same, but instead of using a local
`DISPATCH`, calls go **through the MCP server**. The key detail is that the `inputSchema` field
in an MCP tool definition is already JSON Schema, which is exactly what OpenAI function calling expects.
Therefore, the entire bridge between them is a six-line adapter function.

In [7]:
def mcp_to_openai_tools(mcp_tools):
    """MCP tool definitions -> OpenAI function-tool specs."""
    return [{"type": "function", "function": {
                "name": t.name,
                "description": t.description or "",
                "parameters": t.inputSchema}}
            for t in mcp_tools.tools]

async def run_mcp_agent_local(user_prompt, model=OLLAMA_MODEL, max_steps=5):
    with TemporaryFile("w+", encoding="utf-8") as errlog:
        async with stdio_client(server_hw, errlog=errlog) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()
                tools = mcp_to_openai_tools(await session.list_tools())

                messages = [{"role": "user", "content": user_prompt}]
                for step in range(max_steps):
                    resp = ollama_client.chat.completions.create( # local model
                        model=model, messages=messages, tools=tools, tool_choice="auto")
                    msg = resp.choices[0].message
                    messages.append(msg.model_dump(exclude_none=True))

                    if not msg.tool_calls:
                        return msg.content

                    for tc in msg.tool_calls:
                        args = json.loads(tc.function.arguments)
                        print(f"[step {step}] {tc.function.name}({args})")
                        out = await session.call_tool(tc.function.name, args) # call through MCP
                        messages.append({"role": "tool", "tool_call_id": tc.id,
                                         "content": out.content[0].text})
                return "Stopped: max_steps reached."


print(await run_mcp_agent_local("Is the air in Kyiv safe today?"))

[step 0] get_air_quality({'city': 'Kyiv'})
The air quality in Kyiv is currently considered good with an AQI of 42.


**Expected output:**

```
[step 0] get_air_quality({'city': 'Kyiv'})
Kyiv currently has an AQI of 42, which is in the "good" range — the air is safe today.
```

The model selected `get_air_quality` rather than `get_forecast`, called it through MCP, and
interpreted the result. To give the agent real capabilities such as files, GitHub, or a database,
the client only needs to connect to an appropriate MCP server; the agent code does not change.
This M + N instead of M × N property is a major reason MCP became popular.